All raw data used for the project are available for viewing at https://drive.google.com/drive/folders/1zYKFfSXptnHTim8ERtAOu1GdgunQIPzA?usp=sharing

Installations

In [1]:
!pip install -U pypdfium2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 110.2 MB/s eta 0:00:00


Load datasets into the notebook

In [2]:
# Mounting procedure adapted from https://colab.research.google.com/notebooks/io.ipynb#scrollTo=RWSJpsyKqHjH
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [59]:
import dask.bag as db
import pypdfium2 as pdfium
import re

In [4]:
# @title Book File Names
book_file_names = ['babicka_bozena_nemcova',
'windows8_redakce_businessIT_a_partneri',
'cesky_rozhlas-historie_eva_jesutova_a_kolektiv',
'flvek_05_alois_jirasek',
'flvek_4_alois_jirasek',
'flvek_03_alois_jirasek',
'flvek_02_alois_jirasek',
'flvek_01_alois_jirasek',
'jihoslovanske_jazyky_pavel_krejci',
'nase_nynejsi_krise_tomas_garrigue_masaryk',
'lucerna_alois_jirasek',
'matka_karel_capek',
'basne_josef_vaclav_sladek',
'dalimilova_kronika_dalimil',
'obycejny_zivot_karel_capek',
'povetron_karel_capek',
'hordubal_karel_capek',
'noc_na_karlstejne_jaroslav_vrchlicky',
'pisne_kosmicke_jan_neruda',
'domaci_kucharka_magdalena_dobromila_rettigova',
'hovory_s_tg_masarykem_karel_capek',
'dramaticke_zlomky_karel_hynek_macha',
'povidani_o_pejskovi_a_kocicce_josef_capek',
'obrazy_z_dejin_naroda_ceskeho_iii_vladislav_vancura',
'obrazy_z_dejin_naroda_ceskeho_ii_vladislav_vancura',
'obrazy_z_dejin_naroda_ceskeho_i_vladislav_vancura',
'rur_karel_capek',
'bylo_nas_pet_karel_polacek',
'mistr_kampanus_zikmund_winter',
'konec_starych_casu_vladislav_vancura',
'ballady_a_romance_jan_neruda',
'vec_makropulos_karel_capek',
'krakatit_karel_capek',
'filosofska_historie_alois_jirasek',
'stare_povesti_ceske_alois_jirasek',
'devatero_pohadek_karel_capek',
'kosmuw_letopis_cesky_kosmas',
'tezka_hodina_jiri_wolker',
'host_do_domu_jiri_wolker',
'nova_evropa_tomas_garrigue_masaryk',
'sedm_let_v_jizni_africe_iv_emil_holub',
'sedm_let_v_jizni_africe_iii_emil_holub',
'sedm_let_v_jizni_africe_druha_cesta_emil_holub',
'sedm_let_v_jizni_africe_prvni_cesta_emil_holub',
'bila_nemoc_karel_capek',
'rozmarne_leto_vladislav_vancura',
'maj_karel_hynek_macha',
'kytice_karel_jaromir_erben',
'osudy_dobreho_vojaka_svejka_jaroslav_hasek_volumes3and4',
'osudy_dobreho_vojaka_svejka_jaroslav_hasek_volumes1and2',
'broucci_jan_karafiat',
]

In [50]:
# @title Dictionnary of Abbreviations

#adapted from https://cja.ujc.cas.cz/e-cja/zkratky
abbr_dict = {
    "adj.": "adjektivum",
    "adv.": "adverbium",
    "aj.": "a jiné",
    "akuz.": "akuzativ",
    "apod.": "a podobně",
    "atd.": "a tak dále",
    "býv.": "bývalý",
    "č.": "č",
    "čes.": "český",
    "dat.": "dativ",
    "dolož.": "doloženo",
    "doudl.": "doudlebský",
    "dř.": "dříve",
    "f.": "femininum",
    "gen.": "genitiv",
    "imp.": "imperativ",
    "ind.": "indikativ",
    "inf.": "infinitiv",
    "inform.": "informátor",
    "instr.": "instrumentál",
    "jč.": "jihočeský",
    "již.": "jižní",
    "jjv.": "jihojihovýchodní",
    "jjz.": "jihojihozápadní",
    "jv.": "jihovýchod",
    "jz.": "jihozápad",
    "jzč.": "jihozápadočeský",
    "km": "kilometr",
    "kol.": "kolektiv",
    "kond.": "kondicionál",
    "lid.": "lidový",
    "lok.": "lokál",
    "m.": "maskulinum",
    "m n. m.": "metry nad mořem",
    "min.": "minulý",
    "n. l.": "našeho letopočtu",
    "např.": "například",
    "nar.": "narozen",
    "nář.": "nářečí",
    "nepřízv.": "nepřízvučný",
    "neživ.": "neživotný",
    "nom.": "ominativ",
    "obl.": "oblast",
    "obyv.": "obyvatel",
    "okr.": "okres",
    "os.": "osoba",
    "pl.": "plurál",
    "plt.": "plurale tantum",
    "poč.": "počátek",
    "popř.": "popřípadě",
    "préz.": "prézens",
    "protet.": "protetický",
    "předp.": "předpona",
    "přech.": "přechodník",
    "příč.": "příčestí",
    "příp.": "přípona",
    "přísl.": "příslovce",
    "přít.": "přítomný",
    "přivl.": "přivlastňovací",
    "př. n. l.": "před naším letopočtem",
    "pův.": "původní",
    "r.": "rok",
    "s.": "strana",
    "samohl.": "samohláska",
    "sev.": "severní",
    "sg.": "singulár",
    "slez.": "slezský",
    "souhl.": "souhláska",
    "ssv.": "severoseverovýchodní",
    "ssz.": "severoseverozápadní",
    "stol.": "století",
    "střč.": "středočeský",
    "střm.": "středomoravský",
    "subst.": "substantivum",
    "sv.": "svatý",
    "svč.": "severovýchodočeský",
    "sz.": "severozápad",
    "tj.": "to je",
    "trp.": "trpný",
    "tř.": "třída",
    "tzn.": "to znamená",
    "tzv.": "takzvaný",
    "ukaz.": "ukazovací",
    "vjv.": "východojihovýchodní",
    "vm.": "východomoravský",
    "vok.": "vokativ",
    "vsv.": "východoseverovýchodní",
    "vých.": "východní",
    "zájm.": "zájmeno",
    "záp.": "západní",
    "zč.": "západočeský",
    "zjz.": "západojihozápadní",
    "zsz.": "západoseverozápadní",
    "zvl.": "zvláště",
    "zvrat.": "zvratný",
    "živ.": "životný"
}


['př. n. l.',
 'nepřízv.',
 'inform.',
 'm n. m.',
 'protet.',
 'samohl.',
 'dolož.',
 'doudl.',
 'instr.',
 'neživ.',
 'předp.',
 'přech.',
 'přísl.',
 'přivl.',
 'souhl.',
 'subst.',
 'zvrat.',
 'akuz.',
 'apod.',
 'kond.',
 'n. l.',
 'např.',
 'obyv.',
 'popř.',
 'préz.',
 'příč.',
 'příp.',
 'přít.',
 'slez.',
 'stol.',
 'střč.',
 'střm.',
 'ukaz.',
 'vých.',
 'zájm.',
 'adj.',
 'adv.',
 'atd.',
 'býv.',
 'čes.',
 'dat.',
 'gen.',
 'imp.',
 'ind.',
 'inf.',
 'již.',
 'jjv.',
 'jjz.',
 'jzč.',
 'kol.',
 'lid.',
 'lok.',
 'min.',
 'nar.',
 'nář.',
 'nom.',
 'obl.',
 'okr.',
 'plt.',
 'poč.',
 'pův.',
 'sev.',
 'ssv.',
 'ssz.',
 'svč.',
 'trp.',
 'tzn.',
 'tzv.',
 'vjv.',
 'vok.',
 'vsv.',
 'záp.',
 'zjz.',
 'zsz.',
 'zvl.',
 'živ.',
 'aj.',
 'dř.',
 'jč.',
 'jv.',
 'jz.',
 'os.',
 'pl.',
 'sg.',
 'sv.',
 'sz.',
 'tj.',
 'tř.',
 'vm.',
 'zč.',
 'č.',
 'f.',
 'km',
 'm.',
 'r.',
 's.']

In [68]:
def book_loader(filename):
  document = pdfium.PdfDocument('/content/drive/My Drive/Colab Notebooks/BigDataProject/books/' + filename + '.pdf')

  version_marker_found = 0
  book_started = 0
  text = ''
  for page in document:
    # extract text from the page
    textpage = page.get_textpage()
    extractedtext = textpage.get_text_bounded()

    # the following two conditions ensure that the material attached to the book that is not a part of the original text is skipped (for example the cover page, info about publication etc.)
    if version_marker_found == 0 and 'verze' in extractedtext.lower():
      version_marker_found = 1
      print('version marker found')

    # after the version marker is found (indicating the last page of added material), the next page is checked for containing the contents (obsah) of the book which can also be skipped.
    elif version_marker_found and book_started == 0:
      if 'obsah' not in extractedtext.lower():
        book_started = 1

    if book_started:
      text += extractedtext

  return filename, text

# a function that removes every expression in a list from the given string
def clean_text(text: str, expr: list[str]) -> str:
  for e in expr:
    text = text.replace(e, '')
    print(e)

  return text

def to_sentences(item: tuple):
  name = item[0]
  text = item[1]
  sentences = []

  for sentence in text.split('.'):
    sentences.append((name, sentence))

  return sentences

# converts dictionnary keys to regular expressions that look for the expression followed by non upper case letter character
def to_regex(expr):
  return expr.replace('.', '\\.') + '\\s*[^A-Z]'

def expand_abbr(text):
  for abbr in sorted(abbr_dict, key=len, reverse=True):
    regex = to_regex(abbr)
    print(re.search(regex, text))



In [69]:
expand_abbr('Aristotelés ve svém díle "O nebi" z roku 340 př. n. l. dokázal, že tvar Země musí být kulatý, jelikož stín Země na Měsíci je při zatmění vždy kulatý, což by při plochém tvaru Země nebylo možné.')

None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
<re.Match object; span=(49, 56), match='n. l. d'>
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
<re.Match object; span=(45, 56), match='př. n. l. d'>
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None
None


In [35]:
# load the books
books = db.from_sequence(book_file_names).map(book_loader)

# load the czech wikipedia
wiki = (
    db.read_text('/content/drive/My Drive/Colab Notebooks/BigDataProject/wiki/extracted/extracted/*/*')
    .filter(lambda x: x[:4] != '<doc' and '__NOEDITSECTION__' not in x and '</doc>' not in x)
    .map(lambda x: ('wiki', x))
)

all_text = db.concat([wiki, books])

In [14]:
# verify all files have loaded correctly
print("book partitions", books.npartitions)
print("wiki partitions", wiki.npartitions)

book partitions 51
wiki partitions 1264


In [53]:
print(all_text.take(20))

(('wiki', 'Hlavní strana\n'), ('wiki', '\n'), ('wiki', 'internetové encyklopedii, kterou může .&lt;br&gt;Česká Wikipedie má nyní .\n'), ('wiki', '&lt;br&gt;&lt;br&gt;\n'), ('wiki', ' • • \n'), ('wiki', ' • \n'), ('wiki', 'Ostatní projekty\n'), ('wiki', 'Další informace…\n'), ('wiki', ' • \n'), ('wiki', '. v minulosti\n'), ('wiki', '\n'), ('wiki', 'Astronomie\n'), ('wiki', '\n'), ('wiki', 'Astronomie, řecky αστρονομία z άστρον (astron) hvězda a νόμος (nomos) zákon, česky též hvězdářství, je věda, která se zabývá jevy za hranicemi zemské atmosféry. Zvláště tedy výzkumem vesmírných těles, jejich soustav, různých dějů ve vesmíru i vesmírem jako celkem.\n'), ('wiki', 'Historie astronomie.\n'), ('wiki', 'Antika.\n'), ('wiki', 'Astronomie se podobně jako další vědy začala rozvíjet ve starověku. Na území Babylonie však nebylo k popisu používáno již vynalezené geometrie (grafy). První se z astronomie rozvíjela astrometrie, zabývající se měřením poloh hvězd a planet na obloze. Tato oblast astron

Text cleaning pipeline

In [48]:
# clean the text pipeline
expr_to_clean = ['\n', '\r', ',', '-', '–', '—', ';', '“', '0', '1', '2', '3', '4', '5', '6','7','8','9','\x02', '(', ')', '„', '•', '&ltbr', '&gt']
cleaned_text = (
    all_text.map(lambda x: (x[0], clean_text(x[1], expr_to_clean))) # removes specified expressions
    .map(lambda x: to_sentences(x)).flatten() # splits items into sentences
    .map(lambda x: (x[0], x[1].strip())) # strips trailing spaces
    .filter(lambda x: x[1] != '') # removes empty strings
    .filter(lambda x: len(x[1].split()) >= 3)
)


In [49]:
print(cleaned_text.take(15))

(('wiki', 'internetové encyklopedii kterou může'), ('wiki', 'Česká Wikipedie má nyní'), ('wiki', 'Astronomie řecky αστρονομία z άστρον astron hvězda a νόμος nomos zákon česky též hvězdářství je věda která se zabývá jevy za hranicemi zemské atmosféry'), ('wiki', 'Zvláště tedy výzkumem vesmírných těles jejich soustav různých dějů ve vesmíru i vesmírem jako celkem'), ('wiki', 'Astronomie se podobně jako další vědy začala rozvíjet ve starověku'), ('wiki', 'Na území Babylonie však nebylo k popisu používáno již vynalezené geometrie grafy'), ('wiki', 'První se z astronomie rozvíjela astrometrie zabývající se měřením poloh hvězd a planet na obloze'), ('wiki', 'Tato oblast astronomie měla velký význam pro navigaci'), ('wiki', 'Podstatnou částí astrometrie je sférická astronomie sloužící k popisu poloh objektů na nebeské sféře zavádí souřadnice a popisuje významné křivky a body na nebeské sféře'), ('wiki', 'Pojmy ze sférické astronomie se také používají při měření času'), ('wiki', 'Další oblastí

In [10]:
# todo: expand tzv. to takzvane

In [11]:
print(b.take(1))

NameError: name 'b' is not defined